## Informações disponíveis na véspera e divisão temporal

### Variáveis disponíveis no momento da previsão

A previsão é feita para cada data e tipo de refeição. No momento de prever a demanda da data *t*, podem ser consideradas informações conhecidas até o dia anterior:

- **Calendário:** data prevista, dia da semana, mês e indicador de fim de semana. Essas variáveis são derivadas da data e já são criadas para os conjuntos de treino e teste.
- **Tipo de refeição:** categoria da refeição registrada no mock, incluindo café da manhã, almoço e jantar.
- **Recesso ou férias:** poderá ser usada quando houver um calendário confirmado e disponível antes da previsão. Essa variável não é incluída nesta etapa.
- **Demanda passada:** lags como *t-1* e *t-7* são candidatos para capturar a demanda recente e a sazonalidade semanal. Eles devem ser usados conforme o horizonte de previsão: em uma previsão diária atualizada a cada dia, a demanda observada até *t-1* pode estar disponível; em previsões de vários dias à frente, os lags dos dias futuros precisam ser previstos, e não retirados do conjunto de teste.

Não devem ser usadas como variáveis de entrada informações geradas na própria data prevista, incluindo a demanda real desse dia.

### Divisão entre treino e teste

A base é dividida em ordem cronológica, usando as primeiras 80% das datas para treino e as datas mais recentes para teste. Na execução atual, o treino termina em **18/04/2025** e o teste começa em **19/04/2025**.

A divisão é feita antes de ajustar qualquer etapa de pré-processamento que aprenda parâmetros a partir dos dados. Se forem adicionados imputadores, escalonadores ou codificadores, eles deverão ser ajustados somente com o treino e depois aplicados ao teste. Nesta etapa, o notebook cria a divisão temporal, mas ainda não ajusta esses pré-processadores nem calcula métricas de modelo.

### Observações metodológicas

1. **Pré-processamento:** Caso sejam usados escalonadores, imputadores ou codificadores nas próximas etapas, o ajuste (`fit`) deverá ser realizado exclusivamente com o conjunto de treino. Em seguida, a transformação (`transform`) será aplicada ao treino e ao teste, evitando vazamento de dados (*data leakage*).

2. **Eventos sem período letivo:** A auditoria mostrou que os eventos sem período compatível se concentram em janeiro, julho e dezembro. Esse padrão é compatível com recessos ou intervalos letivos, mas a associação de cada evento com o calendário de recessos do RU ainda precisa ser validada.

In [2]:
import pandas as pd
from pathlib import Path

# 1. Encontrar a raiz do projeto procurando nos diretórios atuais e ancestrais.
diretorio_atual = Path.cwd().resolve()
raiz_projeto = None

for candidato in [diretorio_atual, *diretorio_atual.parents]:
    arquivo_catraca = candidato / "dados" / "bruto" / "mock" / "catraca_ru.csv"
    if arquivo_catraca.is_file():
        raiz_projeto = candidato
        break

if raiz_projeto is None:
    raise FileNotFoundError(
        "Não encontrei dados/bruto/mock/catraca_ru.csv. "
        f"Diretório atual do notebook: {diretorio_atual}"
    )

pasta_mock = raiz_projeto / "dados" / "bruto" / "mock"
caminho_catraca = pasta_mock / "catraca_ru.csv"
caminho_estudantes = pasta_mock / "estudantes.csv"
caminho_componentes = pasta_mock / "matriculas_componentes.csv"

for caminho in [caminho_catraca, caminho_estudantes, caminho_componentes]:
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

# 2. Carregar os três arquivos brutos.
catraca = pd.read_csv(caminho_catraca, dtype={"id_matricula": "string"})
estudantes = pd.read_csv(
    caminho_estudantes,
    dtype={"id_matricula": "string", "periodo_letivo": "string"},
)
componentes = pd.read_csv(
    caminho_componentes,
    dtype={"id_matricula": "string", "periodo_letivo": "string"},
)

print("Raiz do projeto:", raiz_projeto)
print(f"Linhas brutas na catraca: {len(catraca):,}")

# 3. Preparar datas, IDs e períodos para as junções.
catraca["data_hora_refeicao"] = pd.to_datetime(
    catraca["data_hora_refeicao"], errors="coerce"
)
catraca["data"] = catraca["data_hora_refeicao"].dt.normalize()

estudantes["inicio_periodo_letivo"] = pd.to_datetime(
    estudantes["inicio_periodo_letivo"], errors="coerce"
).dt.normalize()
estudantes["fim_periodo_letivo"] = pd.to_datetime(
    estudantes["fim_periodo_letivo"], errors="coerce"
).dt.normalize()

estudantes["periodo_letivo"] = (
    estudantes["periodo_letivo"].astype("string").str.strip()
)
componentes["periodo_letivo"] = (
    componentes["periodo_letivo"].astype("string").str.strip()
)

# Não continuar silenciosamente se faltarem datas ou identificadores.
if catraca["data"].isna().any():
    raise ValueError("Há datas inválidas em data_hora_refeicao.")
if catraca["id_registro_utilizacao"].duplicated().any():
    raise ValueError("id_registro_utilizacao tem duplicatas na catraca.")

# 4. Manter somente refeições servidas.
servidas = catraca.loc[
    catraca["situacao_evento"].eq("servida")
].copy()

print("\nEventos por situação:")
print(catraca["situacao_evento"].value_counts(dropna=False).to_string())
print(f"Eventos servidos: {servidas['id_registro_utilizacao'].nunique():,}")

# 5. Associar cada evento ao período do estudante que cobre a data.
# O processamento em blocos reduz o pico de memória.
colunas_periodo = [
    "id_matricula",
    "periodo_letivo",
    "inicio_periodo_letivo",
    "fim_periodo_letivo",
]

periodos_estudantes = (
    estudantes[colunas_periodo]
    .dropna(subset=["id_matricula", "periodo_letivo"])
    .drop_duplicates()
)

eventos_minimos = servidas[
    ["id_registro_utilizacao", "id_matricula", "data"]
].copy()

associacoes_unicas = []
eventos_ambiguos = set()
tamanho_bloco = 10_000

for inicio in range(0, len(eventos_minimos), tamanho_bloco):
    bloco = eventos_minimos.iloc[inicio : inicio + tamanho_bloco]

    candidatos = bloco.merge(
        periodos_estudantes,
        on="id_matricula",
        how="inner",
    )

    vigentes = candidatos.loc[
        candidatos["data"].between(
            candidatos["inicio_periodo_letivo"],
            candidatos["fim_periodo_letivo"],
            inclusive="both",
        ),
        ["id_registro_utilizacao", "periodo_letivo"],
    ].drop_duplicates()

    periodos_por_evento = (
        vigentes.groupby("id_registro_utilizacao")["periodo_letivo"]
        .nunique()
    )

    ids_ambiguos_bloco = periodos_por_evento[
        periodos_por_evento > 1
    ].index
    eventos_ambiguos.update(ids_ambiguos_bloco.tolist())

    ids_com_um_periodo = periodos_por_evento[
        periodos_por_evento == 1
    ].index

    associacoes_unicas.append(
        vigentes.loc[
            vigentes["id_registro_utilizacao"].isin(ids_com_um_periodo)
        ].drop_duplicates("id_registro_utilizacao")
    )

if associacoes_unicas:
    periodo_por_evento = pd.concat(
        associacoes_unicas,
        ignore_index=True,
    )
else:
    periodo_por_evento = pd.DataFrame(
        columns=["id_registro_utilizacao", "periodo_letivo"]
    )

# Reanexar o período à tabela de eventos servidos.
eventos = servidas.merge(
    periodo_por_evento,
    on="id_registro_utilizacao",
    how="left",
    validate="one_to_one",
)

print("\n=== AUDITORIA DA ASSOCIAÇÃO TEMPORAL ===")
print(f"Eventos servidos: {eventos['id_registro_utilizacao'].nunique():,}")
print(
    "Eventos com período compatível:",
    periodo_por_evento["id_registro_utilizacao"].nunique(),
)
print("Eventos com mais de um período compatível:", len(eventos_ambiguos))
print(
    "Eventos sem período único compatível:",
    eventos.loc[
        eventos["periodo_letivo"].isna(),
        "id_registro_utilizacao",
    ].nunique(),
)

sem_periodo = eventos.loc[eventos["periodo_letivo"].isna()].copy()

print("\n=== EVENTOS SEM PERÍODO POR ANO E MÊS ===")
print(
    sem_periodo.assign(
        ano=sem_periodo["data"].dt.year,
        mes=sem_periodo["data"].dt.month,
    )
    .groupby(["ano", "mes"])
    .size()
    .to_string()
)

print("\n=== AMOSTRA DE DATAS SEM PERÍODO (TOP 20) ===")
print(
    sem_periodo.groupby("data")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .to_string()
)

# 6. Resumir componentes por matrícula e período ANTES da junção.
# Assim, cada chave de matrícula/período aparece no máximo uma vez.
componentes_periodo = (
    componentes.groupby(["id_matricula", "periodo_letivo"])
    .agg(
        quantidade_componentes=("id_matricula_componente", "nunique")
    )
    .reset_index()
)

eventos = eventos.merge(
    componentes_periodo,
    on=["id_matricula", "periodo_letivo"],
    how="left",
    validate="many_to_one",
)

print(
    "\nEventos únicos após associação com componentes:",
    eventos["id_registro_utilizacao"].nunique(),
)

# 7. Criar a demanda diária contando IDs de evento distintos.
base_diaria = (
    eventos.groupby(["data", "tipo_refeicao"])["id_registro_utilizacao"]
    .nunique()
    .reset_index(name="quantidade_servida")
    .sort_values(["data", "tipo_refeicao"])
)

total_esperado = servidas["id_registro_utilizacao"].nunique()
total_agregado = int(base_diaria["quantidade_servida"].sum())

if total_agregado != total_esperado:
    raise ValueError(
        f"Total agregado ({total_agregado:,}) diferente "
        f"do total de eventos servidos ({total_esperado:,})."
    )

# 8. Completar a grade de datas e refeições com zero.
# Isso pressupõe que dias sem evento no mock representam demanda zero.
datas = pd.date_range(
    start=servidas["data"].min(),
    end=servidas["data"].max(),
    freq="D",
)
refeicoes = sorted(servidas["tipo_refeicao"].dropna().unique())

grade = pd.MultiIndex.from_product(
    [datas, refeicoes],
    names=["data", "tipo_refeicao"],
)

base_diaria_completa = (
    base_diaria
    .set_index(["data", "tipo_refeicao"])
    .reindex(grade, fill_value=0)
    .reset_index()
)

base_diaria_completa["quantidade_servida"] = (
    base_diaria_completa["quantidade_servida"].astype("int64")
)

print("\n=== BASE DIÁRIA COMPLETA ===")
print(f"Datas: {len(datas):,}")
print(f"Tipos de refeição: {len(refeicoes)}")
print(f"Linhas: {len(base_diaria_completa):,}")
print(
    "Total de refeições:",
    f"{base_diaria_completa['quantidade_servida'].sum():,}",
)
print(base_diaria_completa.head(10).to_string(index=False))

# Salvar a base com as três colunas da série diária.
pasta_processado = raiz_projeto / "dados" / "processado"
pasta_processado.mkdir(parents=True, exist_ok=True)

caminho_saida = pasta_processado / "base_analitica_diaria.csv"
base_diaria_completa.to_csv(caminho_saida, index=False)

print("\nBase salva em:", caminho_saida)

# 9. Separar treino e teste por data, antes de ajustar qualquer
# pré-processador que aprenda parâmetros a partir dos dados.
datas_ordenadas = (
    base_diaria_completa["data"].drop_duplicates().sort_values().to_list()
)

indice_corte = int(len(datas_ordenadas) * 0.8)
if indice_corte <= 0 or indice_corte >= len(datas_ordenadas):
    raise ValueError("Não há datas suficientes para criar treino e teste.")

data_corte = datas_ordenadas[indice_corte]

treino = base_diaria_completa.loc[
    base_diaria_completa["data"] < data_corte
].copy()

teste = base_diaria_completa.loc[
    base_diaria_completa["data"] >= data_corte
].copy()

# Estas variáveis de calendário são conhecidas na data da previsão.
# Não usamos calendário de recessos porque ainda precisa ser validado.
for conjunto in [treino, teste]:
    conjunto["dia_semana"] = conjunto["data"].dt.day_of_week
    conjunto["mes"] = conjunto["data"].dt.month
    conjunto["fim_de_semana"] = (
        conjunto["dia_semana"] >= 5
    ).astype("int8")

print("\n=== DIVISÃO TEMPORAL TREINO/TESTE ===")
print("Primeira data de teste:", data_corte)
print("Última data de treino:", treino["data"].max())
print("Linhas de treino:", f"{len(treino):,}")
print("Linhas de teste:", f"{len(teste):,}")

if treino["data"].max() >= teste["data"].min():
    raise ValueError("A divisão temporal ficou sobreposta.")

Raiz do projeto: C:\Users\biala\OneDrive\Documentos\entrega-04-lavine0524\previsao-demanda-ru-ufpb
Linhas brutas na catraca: 224,880

Eventos por situação:
situacao_evento
servida    221600
negada       3280
Eventos servidos: 221,600

=== AUDITORIA DA ASSOCIAÇÃO TEMPORAL ===
Eventos servidos: 221,600
Eventos com período compatível: 205823
Eventos com mais de um período compatível: 0
Eventos sem período único compatível: 15777

=== EVENTOS SEM PERÍODO POR ANO E MÊS ===
ano   mes
2019  1      871
      7      850
      12     286
2020  1      847
      7      853
      12     297
2021  1      805
      7      838
      12     320
2022  1      819
      7      813
      12     312
2023  1      826
      7      795
      12     294
2024  1      840
      7      857
      12     288
2025  1      785
      7      934
      12     352
2026  1      952
      7      943

=== AMOSTRA DE DATAS SEM PERÍODO (TOP 20) ===
data
2025-07-02    52
2025-07-22    47
2023-12-29    46
2025-12-24    45
2021-0